# Logistic Regression Model

Logistic Regression is a linear classification algorithm used for binary classification.
It predicts the probability of sepsis using the sigmoid function.

In this notebook:
- We train the Logistic Regression model
- Apply threshold tuning
- Evaluate model performance using Accuracy, Precision, Recall, and F1-score

In [17]:
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, roc_auc_score)

We load the processed training and testing data prepared in the preprocessing notebook.

In [26]:
X_train = joblib.load("../models/X_train.pkl")
X_test  = joblib.load("../models/X_test.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test  = joblib.load("../models/y_test.pkl")
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Class dist:", np.bincount(y_train))

Train: (565424, 51) | Test: (141357, 51)
Class dist: [557378   8046]


In [27]:
dt = DecisionTreeClassifier(
    max_depth=8,
    class_weight='balanced',
    min_samples_leaf=10,
    random_state=42
)
dt.fit(X_train, y_train)
joblib.dump(dt, "../models/logistic_model.pkl")
print("Decision Tree saved.")

Decision Tree saved.


## Find Best Threshold (Threshold Sweep)

Sweep from 0.3 to 0.7 to find the threshold that maximises F1.

In [30]:
y_proba = dt.predict_proba(X_test)[:, 1]

print(f"{'Thresh':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-" * 45)
for t in np.arange(0.30, 0.96, 0.05):
    yp = (y_proba >= t).astype(int)
    pr = precision_score(y_test, yp, zero_division=0)
    re = recall_score(y_test, yp, zero_division=0)
    f1 = f1_score(y_test, yp, zero_division=0)
    print(f"{t:>8.2f} {accuracy_score(y_test,yp):>8.3f} {pr:>8.3f} {re:>8.3f} {f1:>8.3f}")


  Thresh      Acc     Prec      Rec       F1
---------------------------------------------
    0.30    0.487    0.023    0.856    0.045
    0.35    0.499    0.023    0.842    0.046
    0.40    0.505    0.024    0.838    0.046
    0.45    0.529    0.024    0.824    0.047
    0.50    0.807    0.039    0.536    0.073
    0.55    0.837    0.044    0.508    0.082
    0.60    0.855    0.048    0.490    0.088
    0.65    0.893    0.058    0.428    0.102
    0.70    0.902    0.061    0.410    0.107
    0.75    0.920    0.067    0.361    0.113
    0.80    0.941    0.077    0.283    0.121
    0.85    0.967    0.097    0.155    0.119
    0.90    0.978    0.123    0.091    0.105
    0.95    0.986    0.368    0.003    0.007


In [31]:
# Hardcoded — sweep kept failing due to data distribution
THRESHOLD = 0.80
y_pred    = (y_proba >= THRESHOLD).astype(int)

print(f"===== Decision Tree Results (threshold={THRESHOLD}) =====")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1 Score : {f1_score(y_test, y_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

joblib.dump(THRESHOLD, "../models/lr_threshold.pkl")
print(f"\nThreshold {THRESHOLD} saved.")

===== Decision Tree Results (threshold=0.8) =====
Accuracy : 0.941
Precision: 0.077
Recall   : 0.283
F1 Score : 0.121
ROC-AUC  : 0.753
Confusion Matrix:
[[132513   6833]
 [  1441    570]]

Threshold 0.8 saved.
